In [43]:
import pandas as pd
import json
from pathlib import Path


In [ ]:
DATA_DIR = Path("D:/AI_BRCA/DATOS_CONTROL")
METADATA_FILE = DATA_DIR / "metadata.control.json"

In [ ]:
# Leemos el JSON para asociar el nombre del archivo TSV con el 'case submitter id' (ej. TCGA-A1-A0SB-01A)
with open(METADATA_FILE, 'r') as f:
    metadata = json.load(f)

file_to_sample = {}
for item in metadata:
    filename = item.get("file_name")
    # Extraemos el ID del submitter. Tomamos los primeros 15 caracteres para identificar el Sample (ej. TCGA-XX-XXXX-01A)
    # Si quieres el paciente a nivel global sin importar el tipo de muestra, usa [:12] (ej. TCGA-XX-XXXX)
    sample_id = item["associated_entities"][0]["entity_submitter_id"][:15]
    file_to_sample[filename] = sample_id


# A chequear:
    # Algunos 'gene_name' pueden estar duplicados debido a múltiples Ensembl IDs apuntando al mismo símbolo HGNC.
    # Para Machine Learning / DEA, lo estándar es sumar estos conteos.
    df_grouped = df.groupby('gene_name')[['unstranded', 'tpm_unstranded']].sum()

In [46]:
counts_data = {}
tpm_data = {}

# Buscamos de forma recursiva todos los archivos TSV en la carpeta y subcarpetas
tsv_files = list(DATA_DIR.rglob("*.tsv"))
print(f"Encontrados {len(tsv_files)} archivos TSV. Procesando...")

for tsv_path in tsv_files:
    filename = tsv_path.name
    
    # Ignoramos archivos TSV que no estén en nuestro metadata
    if filename not in file_to_sample:
        continue
        
    sample_id = file_to_sample[filename]
    
    # Leer el TSV. Los archivos GDC suelen saltarse los comentarios si existen con comment='#'
    # Las columnas estándar son: gene_id, gene_name, gene_type, unstranded, stranded_first, stranded_second, tpm_unstranded, etc.
    df = pd.read_csv(tsv_path, sep='\t', comment='#')
    
    # Filtrar metadatos del secuenciador que GDC pone al inicio (N_unmapped, N_multimapping, etc.)
    # Nos quedamos solo con las filas donde el gene_id empiece con 'ENSG'
    df = df[df['gene_id'].str.startswith('ENSG', na=False)]
    
    # Algunos 'gene_name' pueden estar duplicados debido a múltiples Ensembl IDs apuntando al mismo símbolo HGNC.
    # Para Machine Learning / DEA, lo estándar es sumar estos conteos.
    df_grouped = df.groupby('gene_name')[['unstranded', 'tpm_unstranded']].sum()
    
    # Guardamos las Series en nuestros diccionarios usando el sample_id como llave
    counts_data[sample_id] = df_grouped['unstranded']
    tpm_data[sample_id] = df_grouped['tpm_unstranded']

Encontrados 113 archivos TSV. Procesando...


In [47]:
# pd.DataFrame(dict) pone las llaves (samples) como columnas y los índices (genes) como filas.
# Usamos .T (Transpuesta) porque en ML y PyDESeq2 se estila: Filas = Samples, Columnas = Genes/Features.
df_counts = pd.DataFrame(counts_data).T
df_tpm = pd.DataFrame(tpm_data).T

# Rellenamos posibles NaNs con 0 en caso de que algún gen falte en alguna muestra
df_counts = df_counts.fillna(0).astype(int)
df_tpm = df_tpm.fillna(0.0)

In [48]:
df_counts.to_csv("../data/processed/tcga_brca_raw_counts_CONTROL.csv")
df_tpm.to_csv("../data/processed/tcga_brca_tpm_CONTROL.csv")

In [50]:
vis = pd.read_csv("../data/processed/tcga_brca_raw_counts_CONTROL.csv", index_col=0)
vis.head()


,5S_rRNA,5_8S_rRNA,7SK,A1BG,A1BG-AS1,A1CF,A2M,A2M-AS1,A2ML1,A2ML1-AS1,...,ZYG11A,ZYG11AP1,ZYG11B,ZYX,ZYXP1,ZZEF1,ZZZ3,hsa-mir-1253,hsa-mir-423,snoZ196
TCGA-BH-A18V-11,2,0,0,14,70,0,32579,94,11,0,...,77,0,1363,13013,0,2071,1842,0,0,0
TCGA-BH-A0BM-11,5,0,0,8,69,4,63528,72,226,0,...,94,0,2460,18776,0,3514,3736,0,0,0
TCGA-BH-A1F2-11,4,0,0,20,100,5,130571,349,66,0,...,116,0,6656,14341,0,8078,4462,0,0,4
TCGA-BH-A1F8-11,1,0,0,4,50,5,56702,87,70,1,...,85,0,2380,8991,0,2684,2718,0,0,3
TCGA-BH-A18S-11,5,0,1,0,91,2,60951,84,35,0,...,113,0,2379,11155,0,4155,2814,0,0,0


In [51]:
vis1 = pd.read_csv("../data/processed/tcga_brca_tpm_CONTROL.csv", index_col=0)
vis1.head()

,5S_rRNA,5_8S_rRNA,7SK,A1BG,A1BG-AS1,A1CF,A2M,A2M-AS1,A2ML1,A2ML1-AS1,...,ZYG11A,ZYG11AP1,ZYG11B,ZYX,ZYXP1,ZZEF1,ZZZ3,hsa-mir-1253,hsa-mir-423,snoZ196
TCGA-BH-A18V-11,1.1893,0.0,0.0000,0.2343,1.3887,0.0000,345.1638,2.1344,0.1029,0.0000,...,1.0980,0.0,11.1181,266.6216,0.0,8.7253,13.6876,0.0,0.0,0.0000
TCGA-BH-A0BM-11,3.0996,0.0,0.0000,0.1315,1.3439,0.0274,660.7428,1.6049,2.0753,0.0000,...,1.3159,0.0,19.6993,377.6599,0.0,14.5338,27.2537,0.0,0.0,0.0000
TCGA-BH-A1F2-11,1.3484,0.0,0.0000,0.1915,1.1351,0.0199,791.5163,4.5341,0.3532,0.0000,...,0.9465,0.0,31.0652,168.1214,0.0,19.4728,18.9712,0.0,0.0,1.7213
TCGA-BH-A1F8-11,0.9092,0.0,0.0000,0.0964,1.4281,0.0502,864.8966,2.8440,0.9427,0.2132,...,1.7451,0.0,27.9506,265.2188,0.0,16.2802,29.0781,0.0,0.0,3.2485
TCGA-BH-A18S-11,3.8264,0.0,0.2722,0.0000,2.1879,0.0169,782.5805,2.3114,0.3968,0.0000,...,1.9528,0.0,23.5175,276.9799,0.0,21.2143,25.3410,0.0,0.0,0.0000
